In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SCENIC+ PIPELINE — mm10 mouse, non-multiome
# MacBook Pro M4 Max — ARM64 native
# ══════════════════════════════════════════════════════════════════════════════

In [1]:
import sys
print(sys.executable)

/Users/cnbr/miniforge3/envs/scenicplus/bin/python


In [2]:
import importlib.metadata

try:
    print("MACS3:", importlib.metadata.version("MACS3"))
except importlib.metadata.PackageNotFoundError:
    print("MACS3 is not installed in the active environment.")

MACS3 is not installed in the active environment.


In [2]:
import os
import warnings
import pickle
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse
import scanpy as sc
import pkg_resources
import pyranges as pr
warnings.filterwarnings('ignore')

/var/folders/c5/9gnt7c7x7x59ygrp90w3n60r0000gn/T/ipykernel_33218/1011782734.py:9: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [3]:
# ── Paths — update HOME if needed ────────────────────────────────────────────
HOME          = os.path.expanduser("~")
WORK_DIR      = f"{HOME}/Downloads/Seurat_scATAC-seq"
INPUT_DIR     = f"{WORK_DIR}/scenicplus_input"
OUT_DIR       = f"{WORK_DIR}/scenicplus_output"
FRAGMENT_FILE = f"{WORK_DIR}/fragments.tsv.gz"

# cistarget databases — download before running Block 3
# URL: https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/
# Files needed (pick one rankings + one scores):
#   mc_v10_clust_nr_motifs-mm10-500bp-upstream-10species.rankings.feather
#   mc_v10_clust_nr_motifs-mm10-500bp-upstream-10species.scores.feather
#   motifs-v10-nr.mgi-m0.001-o0.0.tbl
CISTARGET_DIR = f"{HOME}/Downloads/cistarget_mm10"   # UPDATE THIS

for d in [OUT_DIR,
          f"{OUT_DIR}/scRNA",
          f"{OUT_DIR}/scATAC/pseudobulk",
          f"{OUT_DIR}/scATAC/macs3_peaks",
          f"{OUT_DIR}/scATAC/models",
          f"{OUT_DIR}/scATAC/qc",
          f"{OUT_DIR}/scenicplus/motif_enrichment"]:
    os.makedirs(d, exist_ok=True)

print("Paths OK")
print(f"Fragment file exists: {os.path.exists(FRAGMENT_FILE)}")

Paths OK
Fragment file exists: True


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 1: Build RNA AnnData from R export
# ════════════════════════════════════════════════════════════════════════════

print("\n=== BLOCK 1: RNA AnnData ===")

# Load sparse matrix exported from R
rna_mat    = scipy.io.mmread(f"{INPUT_DIR}/rna_counts.mtx").T.tocsr()
genes      = pd.read_csv(f"{INPUT_DIR}/rna_gene_names.txt", header=None)[0].values
barcodes   = pd.read_csv(f"{INPUT_DIR}/rna_barcodes.txt",  header=None)[0].values
rna_meta   = pd.read_csv(f"{INPUT_DIR}/rna_metadata.csv",  index_col=0)
rna_umap   = pd.read_csv(f"{INPUT_DIR}/rna_umap.csv",      index_col=0)

adata = sc.AnnData(
    X    = rna_mat,
    obs  = rna_meta,
    var  = pd.DataFrame(index=genes)
)
adata.obs_names = barcodes
adata.obsm['X_umap'] = rna_umap[['UMAP_1','UMAP_2']].values

# Store raw before any normalization
adata.raw = adata.copy()

# Normalize for HVG selection only (SCENIC+ uses raw internally)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3,
                              min_disp=0.5, n_top_genes=3000)

print(adata)
print(adata.obs['cell_type'].value_counts())

adata.write_h5ad(f"{OUT_DIR}/scRNA/adata.h5ad", compression='gzip')
print("RNA h5ad saved")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 2: pycisTopic — ATAC preprocessing
# ════════════════════════════════════════════════════════════════════════════

print("\n=== BLOCK 2: pycisTopic ATAC preprocessing ===")

from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
from pycisTopic.lda_models import run_cgs_models, evaluate_models
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.diff_features import (impute_accessibility,
                                       normalize_scores,
                                       find_highly_variable_features,
                                       find_diff_features)
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk, peak_calling
from pycisTopic.iterative_peak_calling import get_consensus_peaks
from pycisTopic.qc import compute_qc_stats

# ── 2a. Chromosome sizes (mm10) ────────────────────────────────────────────
# Download locally to avoid network calls during run
chromsizes_path = f"{OUT_DIR}/mm10.chrom.sizes"
if not os.path.exists(chromsizes_path):
    import urllib.request
    urllib.request.urlretrieve(
        "http://hgdownload.cse.ucsc.edu/goldenPath/mm10/bigZips/mm10.chrom.sizes",
        chromsizes_path
    )
chromsizes = pd.read_csv(chromsizes_path, sep='\t', header=None,
                          names=['Chromosome','End'])
chromsizes['Start'] = 0
chromsizes = chromsizes[
    chromsizes['Chromosome'].str.match(r'^chr(\d{1,2}|X|Y)$')
].reset_index(drop=True)
chromsizes = pr.PyRanges(chromsizes[['Chromosome','Start','End']])
print(f"Chromosomes: {len(chromsizes.chromosomes)}")

In [ ]:
# ── 2b. Blacklist ─────────────────────────────────────────────────────────
BLACKLIST_PATH = f"{OUT_DIR}/mm10-blacklist.v2.bed.gz"
if not os.path.exists(BLACKLIST_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Boyle-Lab/Blacklist/master/lists/mm10-blacklist.v2.bed.gz",
        blacklist_path
    )
print(f"Blacklist: {os.path.exists(BLACKLIST_PATH)}")

In [ ]:
# ── 2c. Cell metadata ─────────────────────────────────────────────────────
cell_data = pd.read_csv(f"{INPUT_DIR}/atac_metadata.csv", index_col=0)
cell_data['cell_type'] = cell_data['cell_type'].astype(str)
cell_data['sample_id'] = "male_cortex"

# CRITICAL for non-multiome: strip -1 suffix from barcodes if present
# so they match the fragment file barcodes exactly
# Check first barcode in fragment file:
import gzip
with gzip.open(FRAGMENT_FILE, 'rt') as f:
    for line in f:
        if not line.startswith('#'):
            first_frag_bc = line.split('\t')[3]
            break
print(f"First fragment barcode: {first_frag_bc}")
print(f"First cell barcode in metadata: {cell_data.index[0]}")
# If they differ (e.g. fragment has no -1), strip from metadata:
# cell_data.index = cell_data.index.str.replace('-1$', '', regex=True)

print(f"ATAC cells: {len(cell_data)}")
print(cell_data['cell_type'].value_counts())

In [2]:
# ── 2d. Pseudobulk peak calling with MACS3 ───────────────────────────────
print("\nRunning pseudobulk peak calling with MACS3...")

import subprocess, os, pickle, re
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk, peak_calling
from pycisTopic.iterative_peak_calling import get_consensus_peaks

os.makedirs(f"{OUT_DIR}/scATAC/pseudobulk/bed", exist_ok=True)
os.makedirs(f"{OUT_DIR}/scATAC/pseudobulk/bw",  exist_ok=True)
os.makedirs(f"{OUT_DIR}/scATAC/macs3_peaks",     exist_ok=True)

macs_check = subprocess.run(['macs3', '--version'], capture_output=True, text=True)
print(f"MACS3: {macs_check.stdout.strip() or macs_check.stderr.strip()}")


Running pseudobulk peak calling with MACS3...


/Users/cnbr/miniforge3/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-22 19:17:28,161	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


NameError: name 'OUT_DIR' is not defined

In [ ]:
# ── 2d-i. Export pseudobulk bed + bigwig files ────────────────────────────────
bw_paths, bed_paths = export_pseudobulk(
    input_data        = cell_data,
    variable          = 'cell_type',
    sample_id_col     = 'sample_id',
    chromsizes        = chromsizes,
    bed_path          = f"{OUT_DIR}/scATAC/pseudobulk/bed/",
    bigwig_path       = f"{OUT_DIR}/scATAC/pseudobulk/bw/",
    path_to_fragments = {"male_cortex": FRAGMENT_FILE},
    n_cpu             = 4,
    normalize_bigwig  = True,
    split_pattern     = '-'
)

pickle.dump(bed_paths, open(f"{OUT_DIR}/scATAC/bed_paths.pkl", 'wb'))
pickle.dump(bw_paths,  open(f"{OUT_DIR}/scATAC/bw_paths.pkl",  'wb'))
print(f"Pseudobulk done: {len(bed_paths)} cell types")

In [ ]:
# ── 2d-ii. Sanitize keys: spaces AND slashes → underscores for MACS3 ─────────
def sanitize_name(name):
    return re.sub(r'[ /()]+', '_', name).strip('_')

ct_name_map_fwd = {ct: sanitize_name(ct) for ct in bed_paths.keys()}
ct_name_map_rev = {v: k for k, v in ct_name_map_fwd.items()}

print("Name mapping:")
for orig, clean in ct_name_map_fwd.items():
    print(f"  '{orig}' → '{clean}'")

bed_paths_clean = {}
bed_dir = f"{OUT_DIR}/scATAC/pseudobulk/bed/"

for ct, path in bed_paths.items():
    ct_clean = ct_name_map_fwd[ct]
    new_path = os.path.join(bed_dir, f"{ct_clean}.fragments.tsv.gz")
    if os.path.exists(path) and path != new_path:
        os.rename(path, new_path)
    bed_paths_clean[ct_clean] = new_path

print(f"\nRenamed {len(bed_paths_clean)} bed_paths for MACS3")

In [ ]:
# ── 2d-iii. Run MACS3 peak calling ───────────────────────────────────────────
narrow_peaks_raw = peak_calling(
    macs_path    = 'macs3',
    bed_paths    = bed_paths_clean,
    outdir       = f"{OUT_DIR}/scATAC/macs3_peaks/",
    genome_size  = 'mm',
    n_cpu        = 4,
    input_format = 'BEDPE',
    shift        = 73,
    ext_size     = 146,
    keep_dup     = 'all',
    q_value      = 0.05
)

# ── 2d-iv. Restore original cell type names ───────────────────────────────────
narrow_peaks = {ct_name_map_rev.get(k, k): v for k, v in narrow_peaks_raw.items()}
print(f"Peak calling done: {len(narrow_peaks)} cell types")
print("Sample keys:", list(narrow_peaks.keys())[:3])

pickle.dump(narrow_peaks, open(f"{OUT_DIR}/scATAC/narrow_peaks.pkl", 'wb'))

In [ ]:
# ── 2d-v. Get consensus peaks (blacklist filtered) ────────────────────────────
import pyranges as pr

# Workaround: pycisTopic's cpm() assigns float results back into int64 Score column,
# which pandas >=2.0 rejects. Pre-cast Score to float64 to avoid LossySetitemError.
narrow_peaks_fixed = {}
for k, v in narrow_peaks.items():
    df = v.df.copy()
    if 'Score' in df.columns:
        df['Score'] = df['Score'].astype('float64')
    narrow_peaks_fixed[k] = pr.PyRanges(df)

consensus_peaks = get_consensus_peaks(
    narrow_peaks_fixed,
    peak_half_width   = 250,
    chromsizes        = chromsizes,
    path_to_blacklist = BLACKLIST_PATH
)
consensus_peaks.to_bed(
    path        = f"{OUT_DIR}/scATAC/consensus_regions.bed",
    keep        = True,
    compression = 'infer',
    chain       = False
)
print(f"Consensus peaks: {len(consensus_peaks)}")
pickle.dump(consensus_peaks, open(f"{OUT_DIR}/scATAC/consensus_peaks.pkl", 'wb'))

In [ ]:
# # ── 2e. Create cisTopic object ────────────────────────────────────────────
# from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
# from pycisTopic.lda_models import run_cgs_models, evaluate_models
# from pycisTopic.topic_binarization import binarize_topics
# from pycisTopic.diff_features import impute_accessibility, normalize_scores, \
#     find_highly_variable_features, find_diff_features

# print("\nCreating cisTopic object...")

# cistopic_obj = create_cistopic_object_from_fragments(
#     path_to_fragments = FRAGMENT_FILE,
#     path_to_regions   = f"{OUT_DIR}/scATAC/consensus_regions.bed",
#     path_to_blacklist = BLACKLIST_PATH,           # ← uppercase, consistent with rest
#     valid_bc          = list(cell_data.index),
#     n_cpu             = 1,
#     project           = "male_cortex",
#     split_pattern     = '-'
# )
# cistopic_obj.add_cell_data(cell_data, split_pattern='-')
# print(cistopic_obj)
# pickle.dump(cistopic_obj, open(f"{OUT_DIR}/scATAC/cistopic_obj_raw.pkl", 'wb'))

In [ ]:
# # ── 2f. LDA topic modeling ────────────────────────────────────────────────
# print("\nRunning LDA topic modeling (slow — go get coffee)...")
# os.makedirs(f"{OUT_DIR}/scATAC/models/", exist_ok=True)

# models = run_cgs_models(
#     cistopic_obj,
#     n_topics       = [15, 20, 25, 30, 35, 40],
#     n_cpu          = 8,
#     n_iter         = 500,
#     random_state   = 42,
#     alpha          = 50,
#     alpha_by_topic = True,
#     eta            = 0.1,
#     eta_by_topic   = False,
#     save_path      = f"{OUT_DIR}/scATAC/models/"
# )
# pickle.dump(models, open(f"{OUT_DIR}/scATAC/models/lda_models.pkl", 'wb'))

In [ ]:
# import pickle, os
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import plotly.io as pio
# pio.kaleido.scope.mathjax = None

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"

# # Load individual per-topic model files saved by run_cgs_models
# # instead of the combined pickle which has the StringDtype issue
# import glob

# model_files = sorted(
#     glob.glob(f"{OUT_DIR}/scATAC/models/*.pkl"),
#     key=lambda f: int(os.path.basename(f).replace('LDA_', '').replace('_topics.pkl', '').replace('.pkl',''))
# )
# print("Found model files:", [os.path.basename(f) for f in model_files])

In [10]:
# import pickle, os, glob

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"

# # Just list what's there — no sorting yet
# all_files = glob.glob(f"{OUT_DIR}/scATAC/models/*.pkl")
# for f in all_files:
#     print(os.path.basename(f))

Topic20.pkl
Topic35.pkl
Topic25.pkl
Topic30.pkl
Topic40.pkl
lda_models.pkl
Topic15.pkl


In [ ]:
# import pandas; print(pandas.__version__)   # should print 3.0.2

In [ ]:
# import pickle, os, glob
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots
# import plotly.io as pio
# pio.kaleido.scope.mathjax = None

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"

# # Only grab files that start with "Topic" and end with .pkl
# model_files = sorted(
#     [f for f in glob.glob(f"{OUT_DIR}/scATAC/models/*.pkl")
#      if os.path.basename(f).startswith('Topic')],
#     key=lambda f: int(''.join(filter(str.isdigit, os.path.basename(f))))
# )
# print("Loading:", [os.path.basename(f) for f in model_files])

# n_topics, log_likes, arun, cao, mimno = [], [], [], [], []
# for f in model_files:
#     m = pickle.load(open(f, 'rb'))
#     n_topics.append(m.n_topic)
#     log_likes.append(float(m.metrics.loc['Metric', 'loglikelihood']))
#     arun.append(float(m.metrics.loc['Metric', 'Arun_2010']))
#     cao.append(float(m.metrics.loc['Metric', 'Cao_Juan_2009']))
#     mimno.append(float(m.metrics.loc['Metric', 'Mimno_2011']))
#     print(f"  ✓ {os.path.basename(f)} — {m.n_topic} topics")

# fig = make_subplots(rows=1, cols=4,
#     subplot_titles=['Log-likelihood (↑)', 'Arun 2010 (↓)', 'Cao Juan 2009 (↓)', 'Mimno 2011 (↑)'],
#     horizontal_spacing=0.08)

# for col, (vals, color, name) in enumerate([
#     (log_likes, 'steelblue', 'Log-likelihood'),
#     (arun, 'tomato', 'Arun 2010'),
#     (cao, 'seagreen', 'Cao Juan 2009'),
#     (mimno, 'mediumpurple', 'Mimno 2011'),
# ], start=1):
#     fig.add_trace(go.Scatter(
#         x=n_topics, y=vals, mode='lines+markers', name=name,
#         line=dict(color=color, width=2.5),
#         marker=dict(size=9, color=color, line=dict(width=1.5, color='white')),
#         showlegend=False
#     ), row=1, col=col)
#     fig.add_vline(x=30, line_dash='dash', line_color='gray', line_width=1.5, opacity=0.6, row=1, col=col)

# fig.update_xaxes(tickvals=n_topics, title_text='Number of topics', showgrid=True, gridcolor='#eeeeee')
# fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')
# fig.update_layout(
#     title=dict(text='LDA Model Selection — Male Cortex scATAC', font=dict(size=16), x=0.5, xanchor='center'),
#     height=420, width=1300, plot_bgcolor='white', paper_bgcolor='white',
#     font=dict(family='Arial', size=12), margin=dict(t=80, b=60, l=60, r=30)
# )

# fig.write_image(f"{OUT_DIR}/scATAC/models/model_selection.png", scale=2)
# print("\nPlot saved.")
# fig.show()

In [ ]:
# SELECT_N = 30

# models = pickle.load(open(f"{OUT_DIR}/scATAC/models/lda_models.pkl", 'rb'))
# model = next(m for m in models if m.n_topic == SELECT_N)
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_raw.pkl", 'rb'))
# cistopic_obj.add_LDA_model(model)
# pickle.dump(cistopic_obj, open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'wb'))
# print(f"cisTopic object saved with {SELECT_N}-topic model.")

In [ ]:
# import matplotlib
# matplotlib.use('Agg')  # must be BEFORE any pycisTopic import
# import matplotlib.pyplot as plt
# plt.switch_backend('Agg')

# import pickle, os
# from pycisTopic.topic_binarization import binarize_topics
# from pycisTopic.diff_features import (
#     impute_accessibility, normalize_scores,
#     find_highly_variable_features, find_diff_features
# )

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'rb'))

# region_bin_otsu  = binarize_topics(cistopic_obj, method='otsu',  plot=False)
# region_bin_top3k = binarize_topics(cistopic_obj, method='ntop',  ntop=3000, plot=False)
# pickle.dump(region_bin_otsu,  open(f"{OUT_DIR}/scATAC/region_bin_topics_otsu.pkl",  'wb'))
# pickle.dump(region_bin_top3k, open(f"{OUT_DIR}/scATAC/region_bin_topics_top3k.pkl", 'wb'))
# print("Binarization saved.")

In [ ]:
# # Load cistopic object
# print("Loading cistopic object...")
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'rb'))
# print("Loaded.")

# from pycisTopic.diff_features import (
#     impute_accessibility, normalize_scores,
#     find_highly_variable_features, find_diff_features
# )

# print("Imputing accessibility...")
# imputed_acc = impute_accessibility(
#     cistopic_obj,
#     selected_cells=None,
#     selected_regions=None,
#     scale_factor=10**6
# )
# pickle.dump(imputed_acc, open(f"{OUT_DIR}/scATAC/imputed_acc.pkl", 'wb'))
# print("Imputed accessibility saved.")

In [ ]:
# import matplotlib
# matplotlib.use('Agg')
# import pickle
# import numpy as np
# import pandas as pd

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"

# imputed_acc  = pickle.load(open(f"{OUT_DIR}/scATAC/imputed_acc.pkl", 'rb'))
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'rb'))

# # Inspect what attributes are available
# print(type(imputed_acc))
# print([a for a in dir(imputed_acc) if not a.startswith('_')])
# print("mtx type:", type(imputed_acc.mtx))
# print("mtx shape:", imputed_acc.mtx.shape)

In [ ]:
# print("feature_names length:", len(imputed_acc.feature_names))
# print("cell_names length:",    len(imputed_acc.cell_names))
# print("mtx shape:",            imputed_acc.mtx.shape)

In [ ]:
# print("First 3 feature_names:", list(imputed_acc.feature_names[:3]))
# print("First 3 cell_names:",    list(imputed_acc.cell_names[:3]))

In [ ]:
# print("mtx shape:", imputed_acc.mtx.shape)
# print("feature_names[:3]:", list(imputed_acc.feature_names[:3]))  # rows
# print("cell_names[:3]:",    list(imputed_acc.cell_names[:3]))      # cols?
# print("len feature_names:", len(imputed_acc.feature_names))
# print("len cell_names:",    len(imputed_acc.cell_names))

In [ ]:
# import matplotlib
# matplotlib.use('Agg')
# import pickle
# import numpy as np
# import pandas as pd

# OUT_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"

# imputed_acc  = pickle.load(open(f"{OUT_DIR}/scATAC/imputed_acc.pkl", 'rb'))
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'rb'))

# print("Normalizing (manual)...")
# # mtx is (746029, 13036) dense numpy array — work in float32
# X = imputed_acc.mtx.astype(np.float32)   # in-place dtype conversion

# # Per-cell scaling
# row_sums = X.sum(axis=1, keepdims=True)
# row_sums[row_sums == 0] = 1
# X = X / row_sums * 1e4
# X = np.log1p(X)
# print("Normalization done.")

# # Chunked mean+variance — avoids allocating a second full copy
# print("Computing HVF (chunked)...")
# n_regions  = X.shape[1]
# chunk_size = 2000
# means      = np.zeros(n_regions, dtype=np.float32)
# vars_      = np.zeros(n_regions, dtype=np.float32)

# for start in range(0, n_regions, chunk_size):
#     end = min(start + chunk_size, n_regions)
#     chunk = X[:, start:end]           # view, no copy
#     means[start:end] = chunk.mean(axis=0)
#     # Welford-style: var without allocating a second matrix
#     vars_[start:end]  = chunk.var(axis=0)
#     print(f"  {end}/{n_regions} regions done...")

In [ ]:
# print("Recomputing HVF per region (axis=1)...")
# n_regions  = X.shape[0]   # 746029 rows = regions
# chunk_size = 5000
# means2     = np.zeros(n_regions, dtype=np.float32)
# vars2      = np.zeros(n_regions, dtype=np.float32)

# for start in range(0, n_regions, chunk_size):
#     end = min(start + chunk_size, n_regions)
#     chunk = X[start:end, :]           # (chunk_size x 13036)
#     means2[start:end] = chunk.mean(axis=1)
#     vars2[start:end]  = chunk.var(axis=1)
#     if start % 200000 == 0:
#         print(f"  {start}/{n_regions} done...")

# dispersion2 = np.where(means2 > 0, vars2 / means2, 0)
# top_idx     = np.argsort(dispersion2)[::-1][:5000]
# var_regions = [imputed_acc.feature_names[i] for i in top_idx]  # feature_names = 746029 regions ✓

# print("First 3 var_regions:", var_regions[:3])   # should show chr1:... coords
# print(f"HVF done: {len(var_regions)} regions selected.")
# pickle.dump(var_regions, open(f"{OUT_DIR}/scATAC/var_regions.pkl", 'wb'))
# print("var_regions saved.")

In [ ]:
# from pycisTopic.diff_features import find_diff_features

# markers_dict = find_diff_features(
#     cistopic_obj, imputed_acc,
#     variable      = 'cell_type',
#     var_features  = var_regions,
#     split_pattern = '-',
#     n_cpu         = 2
# )
# pickle.dump(markers_dict, open(f"{OUT_DIR}/scATAC/DARs_cell_type.pkl", 'wb'))
# print("Block 2g complete.")

In [8]:
# import pickle

# var_regions = pickle.load(open(
#     "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output/scATAC/var_regions.pkl", 'rb'
# ))

# print(f"Type:   {type(var_regions)}")
# print(f"Length: {len(var_regions)}")
# print(f"First 5: {var_regions[:5]}")
# print(f"Last 5:  {var_regions[-5:]}")

Type:   <class 'list'>
Length: 5000
First 5: ['chr11:19865410-19865910', 'chr5:33922343-33922843', 'chr4:156064927-156065427', 'chr6:80813528-80814028', 'chr6:80204980-80205480']
Last 5:  ['chr15:55557175-55557675', 'chr8:29142395-29142895', 'chr14:79870261-79870761', 'chr1:161212096-161212596', 'chr15:33651215-33651715']


In [ ]:
# import sys
# print(sys.executable)

In [ ]:
# import subprocess, sys
# result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'scenicplus'], capture_output=True, text=True)
# print(result.stdout or result.stderr)

In [ ]:
# import pycistarget.motif_enrichment_cistarget as mec
# print(dir(mec))

In [ ]:
# # ════════════════════════════════════════════════════════════════════════════
# # BLOCK 3: pycistarget motif enrichment (Updated for pycistarget v1.1+)
# # ════════════════════════════════════════════════════════════════════════════
# import os, re
# import pyranges as pr
# import pycisTopic
# from pycistarget.utils import region_names_to_coordinates
# # New import paths for v1.1+
# from pycistarget.motif_enrichment_cistarget import cisTarget
# from pycistarget.motif_enrichment_result import MotifEnrichmentResult

# OUT_DIR       = "/Users/cnbr/Downloads/Seurat_scATAC-seq/scenicplus_output"
# CISTARGET_DIR = "/Users/cnbr/Downloads/Seurat_scATAC-seq/cistarget_db"

# def sanitize_name(name):
#     return re.sub(r'[ /()]+', '_', name).strip('_')

# import pickle
# import pandas as pd
# import numpy as np

# class LegacyUnpickler(pickle.Unpickler):
#     def find_class(self, module, name):
#         # Intercept the broken Arrow/String types
#         if 'StringDtype' in name or 'ArrowStringArray' in name:
#             return lambda *args, **kwargs: np.array([], dtype=object)
#         return super().find_class(module, name)

# def force_load_to_dataframe(path):
#     """
#     Manually reconstructs the dataframe from the pickle stream 
#     to avoid the broken Index.__new__ calls.
#     """
#     with open(path, 'rb') as f:
#         try:
#             # Try a standard load first
#             return pickle.load(f)
#         except (AttributeError, TypeError):
#             # If it fails, we fall back to reading the raw dict if possible
#             f.seek(0)
#             data = LegacyUnpickler(f).load()
            
#             # If the data comes back as a dict of DataFrames (common in SCENIC+)
#             if isinstance(data, dict):
#                 new_dict = {}
#                 for k, v in data.items():
#                     if isinstance(v, pd.DataFrame):
#                         # Force conversion to standard objects to drop Arrow
#                         new_dict[k] = v.astype(object)
#                     else:
#                         new_dict[k] = v
#                 return new_dict
#             return data

# # Try loading again with the conversion logic
# try:
#     region_bin_otsu  = force_load_to_dataframe(f"{OUT_DIR}/scATAC/region_bin_topics_otsu.pkl")
#     region_bin_top3k = force_load_to_dataframe(f"{OUT_DIR}/scATAC/region_bin_topics_top3k.pkl")
#     markers_dict     = force_load_to_dataframe(f"{OUT_DIR}/scATAC/DARs_cell_type.pkl")
#     print("✅ Success! Data loaded and converted to standard objects.")
# except Exception as e:
#     print(f"❌ Failed even with conversion: {e}")

# region_sets = {'topics_otsu': {}, 'topics_top_3': {}, 'DARs': {}}

# # Format region sets into PyRanges
# for topic, df in region_bin_otsu.items():
#     regions = df.index[df.index.str.startswith('chr')]
#     if len(regions) > 0:
#         region_sets['topics_otsu'][topic] = pr.PyRanges(region_names_to_coordinates(regions))

# for topic, df in region_bin_top3k.items():
#     regions = df.index[df.index.str.startswith('chr')]
#     if len(regions) > 0:
#         region_sets['topics_top_3'][topic] = pr.PyRanges(region_names_to_coordinates(regions))

# for dar, df in markers_dict.items():
#     regions = df.index[df.index.str.startswith('chr')]
#     if len(regions) > 0:
#         region_sets['DARs'][sanitize_name(dar)] = pr.PyRanges(region_names_to_coordinates(regions))

# # DB paths
# db_fnames = [
#     os.path.join(CISTARGET_DIR, "mm10_screen_v10_clust.regions_vs_motifs.rankings.feather"),
#     os.path.join(CISTARGET_DIR, "mm10_screen_v10_clust.regions_vs_motifs.scores.feather"),
# ]
# motif_annotation_fname = os.path.join(CISTARGET_DIR, "motifs-v10-nr.mgi-m0.00001-o0.0.tbl")

# # Verification
# assert all(os.path.exists(f) and os.path.getsize(f) > 1e6 for f in db_fnames), "DB files missing"
# assert os.path.exists(motif_annotation_fname), "Annotation file missing"

# os.makedirs(f"{OUT_DIR}/scenicplus/motif_enrichment/", exist_ok=True)

# menr = {}
# for set_name, sets in region_sets.items():
#     for name, regions in sets.items():
#         print(f"Running pycisTarget on {set_name}/{name} ({len(regions)} regions)...")
        
#         # 1. Run cisTarget search (v1.1 replacement for run_pycistarget)
#         # Note: rankings_db is passed to ctx_db
#         ctx_results = cisTarget(
#             region_sets=regions,
#             ctx_db=db_fnames[0], # Rankings feather
#             ctx_scores=db_fnames[1], # Scores feather
#             n_cpu=4
#         )
        
#         # 2. Calculate enrichment (v1.1 replacement for compute_motif_enrichment)
#         # We initialize the result class and then call its run method
#         enrichment_obj = MotifEnrichmentResult(
#             ctx_results,
#             annotation_fpath=motif_annotation_fname,
#             path_to_motifs=None
#         )
        
#         # This populates the motifs metadata
#         menr[f"{set_name}_{sanitize_name(name)}"] = enrichment_obj

# # Save the results
# with open(f"{OUT_DIR}/scenicplus/motif_enrichment/menr.pkl", 'wb') as f:
#     pickle.dump(menr, f)

# print(f"\nBlock 3 complete. {len(menr)} sets processed.")

In [5]:
# import scenicplus
# import pkgutil

# # List all submodules in scenicplus to find the right name
# package = scenicplus
# for loader, module_name, is_pkg in pkgutil.walk_packages(package.__path__, package.__name__ + "."):
#     if 'umap' in module_name or 'dimension' in module_name or 'DR' in module_name:
#         print(module_name)

In [ ]:
# # ════════════════════════════════════════════════════════════════════════════
# # BLOCK 4: Create SCENIC+ object + run GRN
# # ════════════════════════════════════════════════════════════════════════════
# import pyranges as pr

# print("\n=== BLOCK 4: SCENIC+ GRN ===")

# from scenicplus.scenicplus_class import create_SCENICPLUS_object
# from scenicplus.preprocessing.filtering import apply_std_filtering_to_eRegulons
# from scenicplus.eregulon_enrichment import score_eRegulons
# from scenicplus.DR.utils import run_eRegulons_umap
# from scenicplus.diff_features import get_differential_eRegulons
# from scenicplus.wrappers.run_scenicplus import run_scenicplus

# adata        = sc.read_h5ad(f"{OUT_DIR}/scRNA/adata.h5ad")
# cistopic_obj = pickle.load(open(f"{OUT_DIR}/scATAC/cistopic_obj_lda.pkl", 'rb'))

# scplus_obj = create_SCENICPLUS_object(
#     scRNA_obj              = adata,
#     cisTopic_obj           = cistopic_obj,
#     menr                   = menr,
#     multi_ome_mode         = False,
#     key_to_group_by        = 'cell_type',
#     nr_cells_per_metacells = 5,
#     use_raw_for_scRNAseq_expression_filtering = True
# )
# print(scplus_obj)
# pickle.dump(scplus_obj, open(f"{OUT_DIR}/scenicplus/scplus_obj_initial.pkl", 'wb'))

# os.makedirs(f"{OUT_DIR}/scenicplus/", exist_ok=True)

# run_scenicplus(
#     scplus_obj           = scplus_obj,
#     variable_of_interest = 'cell_type',
#     region_set_key       = 'custom_cistromes',
#     save_path            = f"{OUT_DIR}/scenicplus/",
#     biomart_host         = 'http://www.ensembl.org',
#     species              = 'mmusculus',
#     upstream             = [1000, 150000],
#     downstream           = [1000, 150000],
#     n_cpu                = 4,
#     calculate_TF_RE_correlation = True
# )
# pickle.dump(scplus_obj, open(f"{OUT_DIR}/scenicplus/scplus_obj_grn.pkl", 'wb'))

# # ── Filter + score eRegulons ──────────────────────────────────────────────
# apply_std_filtering_to_eRegulons(scplus_obj)

# score_eRegulons(
#     scplus_obj,
#     ranking_db_fname        = db_fnames[0],
#     eRegulon_signatures_key = 'eRegulon_signatures_filtered',
#     n_cpu                   = 4
# )
# run_eRegulons_umap(scplus_obj, scale=True, signature_keys=['eRegulon_AUC_filtered'])

# # ── Export results ────────────────────────────────────────────────────────
# eregulon_df = scplus_obj.uns['eRegulon_metadata_filtered'].copy()
# eregulon_df.to_csv(f"{OUT_DIR}/scenicplus/eRegulon_summary.csv", index=False)

# auc_df = pd.DataFrame(
#     scplus_obj.obsm['eRegulon_AUC_filtered'],
#     index   = scplus_obj.obs_names,
#     columns = scplus_obj.uns['eRegulon_AUC_filtered_names']
# )
# auc_df['genotype']  = scplus_obj.obs['genotype']
# auc_df['cell_type'] = scplus_obj.obs['cell_type']
# auc_df.to_csv(f"{OUT_DIR}/scenicplus/eRegulon_AUC_per_cell.csv")

# diff_ereg = get_differential_eRegulons(
#     scplus_obj,
#     variable          = 'genotype',
#     contrast_factor_1 = 'KO',
#     contrast_factor_2 = 'Ctrl',
#     split_by          = 'cell_type'
# )
# diff_ereg.to_csv(f"{OUT_DIR}/scenicplus/eRegulon_KOvCtrl_differential.csv", index=False)

# pickle.dump(scplus_obj, open(f"{OUT_DIR}/scenicplus/scplus_obj_final.pkl", 'wb'))
# print("\n=== SCENIC+ complete ===")
# print(f"eRegulons: {len(eregulon_df)}")
# print(f"Differential eRegulons saved to: {OUT_DIR}/scenicplus/")